In [1]:
import yaml

with open("../configs/pretrain_project/silica/baselines/config_pbc_processed_as.yml") as f:
    config = yaml.safe_load(f)

In [2]:
from matdeeplearn.trainers.base_trainer import BaseTrainer

dataset = BaseTrainer._load_dataset(config["dataset"], config["task"]["run_mode"]) if "src" in config["dataset"] else None
model = BaseTrainer._load_model(config["model"], config["dataset"]["preprocess_params"], dataset, 1, 0)[0]
sampler = BaseTrainer._load_sampler(config["optim"], dataset, 1, 0) if "src" in config["dataset"] else None

/net/csefiles/coc-fung-cluster/Qianyu/stable_md/MatDeepLearn_dev/matdeeplearn/preprocessor/datasets.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slic

In [3]:
import torch

checkpoint_pth = "../results/2024-11-01-11-31-44-950-graphormer3d_pbc_gbf_as_no_force_head/checkpoint_0/best_checkpoint.pt"
model.load_state_dict(torch.load(checkpoint_pth, map_location="cpu")["state_dict"])


/tmp/ipykernel_400671/3881077989.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint_pth, map_location="cpu")["state_dict"])


<All keys matched successfully>

In [4]:
from torch_geometric.loader import DataLoader
from matdeeplearn.preprocessor.pbc_transform import Batch

loader = DataLoader(dataset['test'], batch_size=1, shuffle=False)
loader.collate_fn = Batch.from_datalist

In [5]:
import numpy as np
import torch

model = model.to("cuda:0")
model.eval()
batch = next(iter(loader)).to("cuda:0")

starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
repetitions = 5
timings = np.zeros((repetitions,1))

#GPU-WARM-UP
for _ in range(10):
    _ = model(batch)

# MEASURE PERFORMANCE
with torch.no_grad():
    for rep in range(repetitions):
        starter.record()
        for batch in loader:
            _ = model(batch.to("cuda:0"))
        ender.record()
        # WAIT FOR GPU SYNC
        torch.cuda.synchronize()
        curr_time = starter.elapsed_time(ender)
        timings[rep] = curr_time

mean_syn = np.sum(timings) / repetitions
std_syn = np.std(timings)
print(f"Mean inference time: {mean_syn:.2f} ms")
print(f"Std  inference time: {std_syn:.2f} ms")

Mean inference time: 8988.45 ms
Std  inference time: 391.95 ms


In [14]:
for i in range(4):
    print(model.layers[i].self_attn.adaptive_span.get_current_avg_span(), model.layers[i].self_attn.adaptive_span._max_span)

553 2048
305 2048
289 2048
317 2048


In [ ]:
from torch.profiler import profile, record_function, ProfilerActivity
import torch.autograd.profiler as profiler
import torch

model = model.to("cuda:1")
model.eval()

with torch.no_grad():
    for batch in loader:
        try:
            model(batch.to("cuda:1"))
        except:
            continue

with profiler.profile(with_stack=True, profile_memory=True, use_device="cuda") as prof:
    loader_iter = iter(loader)
    for i in range(50):
        batch = next(loader_iter)
        model(batch.to("cuda:1"))

In [ ]:
import pandas as pd
from io import StringIO

table_str = prof.key_averages(group_by_stack_n=5).table(sort_by='cuda_time', row_limit=50)

# Convert the string table to a DataFrame
df = pd.read_csv(StringIO(table_str), sep=r'\s{2,}', engine='python')

In [ ]:
df.columns = df.iloc[0]
df = df.iloc[2:].reset_index(drop=True)
df.to_excel('profiler_full_attn.xlsx')